# Mini Dataset de Visao Computacional  — Processamento de Imagens Digitais

**Disciplina:** Processamento de Imagens Digitais  
**Atividade:** Construcao de Mini Dataset com 2 Classes  
**Dispositivo de captura:** Smartphone Samsung A12  
**Condicao de iluminacao:** Noturna (luz artificial interna)  
**Distancia aproximada:** ~15-20 cm dos objetos  
**Observacao:** Imagens capturadas sem filtros automaticos, sem embelezamento. Leve ruido visivel em algumas imagens devido a iluminacao noturna e ISO elevado.

---

## 1. Definicao das Classes

| Classe | Descricao |
|--------|-----------|
| **classe_A** | Objetos com cor dominante **rosa** |
| **classe_B** | Objetos com cor dominante **nao rosa** (azul, verde, etc.) |

Cada classe contem 5 imagens originais, totalizando **10 imagens**.

---

## 2. Registro de Aquisicao

- **Dispositivo:** Samsung A12 (camera traseira 48MP, utilizada em modo automatico)
- **Condicao de iluminacao:** Noturna, com luz artificial de abajur e luz fluorescente do ambiente
- **Distancia aproximada:** Entre 10 e 25 cm do objeto
- **Angulos:** Variacao entre frontal, leve inclinacao (~30 graus) e zenital (~45 graus)
- **Observacoes de ruido/desfoque:** Em algumas imagens nota-se leve ruido granular (devido ao ISO automatico em ambiente escuro) e leve desfoque em regioes perifericas por nao estabilizacao perfeita das maos.

## 3. Organizacao do Mini Dataset

Estrutura final do dataset:

```
mini_dataset/
├── classe_A_rosa/
│   ├── img001_original.jpg
│   ├── img002_res_50pct.jpg
│   ├── img003_res_20pct.jpg
│   ├── img004_hsv.jpg
│   ├── img005_gray.jpg
│   ├── img006_q256.jpg
│   ├── img007_q64.jpg
│   ├── img008_q32.jpg
│   ├── img009_q2.jpg
│   ├── img010_jpeg_comprimido.jpg
│   └── img011_png_semperda.png
├── classe_B_nao_rosa/
│   ├── img001_original.jpg
│   ├── img002_res_50pct.jpg
│   ├── img003_res_20pct.jpg
│   ├── img004_hsv.jpg
│   ├── img005_gray.jpg
│   ├── img006_q256.jpg
│   ├── img007_q64.jpg
│   ├── img008_q32.jpg
│   ├── img009_q2.jpg
│   ├── img010_jpeg_comprimido.jpg
│   └── img011_png_semperda.png
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path


def find_dataset_root() -> Path:
    """Encontra a pasta do projeto que contem dataset/classe_A e dataset/classe_B."""
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Documents" / "processamento-imagem",
        Path(r"C:\Users\Public\processamento-imagem"),
    ]
    for base in candidates:
        try:
            base = base.resolve()
        except OSError:
            continue
        if (base / "dataset" / "classe_A").is_dir() and (base / "dataset" / "classe_B").is_dir():
            return base
    return Path.cwd().resolve()


NOTEBOOK_ROOT = find_dataset_root()
DATASET_ORIG = NOTEBOOK_ROOT / "dataset"
OUTPUT_DIR = NOTEBOOK_ROOT / "mini_dataset"
CLASSE_A_DIR = DATASET_ORIG / "classe_A"
CLASSE_B_DIR = DATASET_ORIG / "classe_B"

# Criar diretorios de saida
for subdir in ["classe_A_rosa", "classe_B_nao_rosa"]:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", NOTEBOOK_ROOT)
print("Diretorios criados com sucesso!")
print("Origem classe A:", CLASSE_A_DIR)
print("Origem classe B:", CLASSE_B_DIR)

In [ ]:
def list_images(folder: Path) -> list[Path]:
    """Lista imagens na pasta (jpg/jpeg/png, qualquer capitalizacao)."""
    if not folder.is_dir():
        raise FileNotFoundError(f"Pasta nao encontrada: {folder}")
    patterns = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    files: list[Path] = []
    for pat in patterns:
        files.extend(folder.glob(pat))
    return sorted({p.resolve() for p in files}, key=lambda p: p.name.lower())


# Listar imagens originais
img_a = list_images(CLASSE_A_DIR)
img_b = list_images(CLASSE_B_DIR)

print("Imagens classe A:", [f.name for f in img_a])
print("Imagens classe B:", [f.name for f in img_b])

if not img_a or not img_b:
    raise RuntimeError(
        "E necessario pelo menos 1 imagem em cada classe. Verifique:\n"
        f"  {CLASSE_A_DIR}\n  {CLASSE_B_DIR}"
    )

# Carregar imagens
imgs_a = [cv2.imread(str(p)) for p in img_a]
imgs_b = [cv2.imread(str(p)) for p in img_b]

for classe, paths, arrs in [("A", img_a, imgs_a), ("B", img_b, imgs_b)]:
    for p, arr in zip(paths, arrs):
        if arr is None:
            raise RuntimeError(f"Falha ao ler imagem: {p}")

print(f"\nImagem classe_A[0] shape: {imgs_a[0].shape}")
print(f"Imagem classe_B[0] shape: {imgs_b[0].shape}")

## 4. A. Analise de Resolucao

A resolucao de uma imagem determina a quantidade de detalhes que ela contem. Reduzir a resolucao implica perder informacao espacial, o que pode afetar diretamente o desempenho de algoritmos de visao computacional (deteccao de bordas, reconhecimento de padroes, classificacao).

**Procedimento:**
- Manter resolucao original
- Reduzir para 50% da resolucao original
- Reduzir para 20% da resolucao original

**Impacto esperado:** Quanto menor a resolucao, mais detalhes sao perdidos. Em tarefas como deteccao de objetos pequenos, a perda de resolucao pode tornar o objeto indetectavel. Em classificacao de cenas, pode haver perda de texturas finas.

In [ ]:
# Analise de Resolucao — 1 imagem de cada classe
img_a_orig = imgs_a[0].copy()
img_b_orig = imgs_b[0].copy()

def resize_image(img, scale_pct):
    h, w = img.shape[:2]
    new_w = int(w * scale_pct / 100)
    new_h = int(h * scale_pct / 100)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

# Variações de resolucao
img_a_50 = resize_image(img_a_orig, 50)
img_a_20 = resize_image(img_a_orig, 20)
img_b_50 = resize_image(img_b_orig, 50)
img_b_20 = resize_image(img_b_orig, 20)

# Salvar imagens
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img001_original.jpg"), img_a_orig)
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img002_res_50pct.jpg"), img_a_50)
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img003_res_20pct.jpg"), img_a_20)
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img001_original.jpg"), img_b_orig)
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img002_res_50pct.jpg"), img_b_50)
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img003_res_20pct.jpg"), img_b_20)

print("Imagens de resolucao salvas com sucesso!")

# Exibir resultados
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(cv2.cvtColor(img_a_orig, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Original (" + str(img_a_orig.shape[1]) + "x" + str(img_a_orig.shape[0]) + ")")
axes[0, 0].axis("off")

axes[0, 1].imshow(cv2.cvtColor(img_a_50, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("50pct Res (" + str(img_a_50.shape[1]) + "x" + str(img_a_50.shape[0]) + ")")
axes[0, 1].axis("off")

axes[0, 2].imshow(cv2.cvtColor(img_a_20, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title("20pct Res (" + str(img_a_20.shape[1]) + "x" + str(img_a_20.shape[0]) + ")")
axes[0, 2].axis("off")

axes[1, 0].imshow(cv2.cvtColor(img_b_orig, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Original (" + str(img_b_orig.shape[1]) + "x" + str(img_b_orig.shape[0]) + ")")
axes[1, 0].axis("off")

axes[1, 1].imshow(cv2.cvtColor(img_b_50, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("50pct Res (" + str(img_b_50.shape[1]) + "x" + str(img_b_50.shape[0]) + ")")
axes[1, 1].axis("off")

axes[1, 2].imshow(cv2.cvtColor(img_b_20, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("20pct Res (" + str(img_b_20.shape[1]) + "x" + str(img_b_20.shape[0]) + ")")
axes[1, 2].axis("off")

plt.suptitle("Analise de Resolucao - Comparacao Visual", fontsize=16)
plt.tight_layout()
plt.show()

# Informacoes tecnicas
print("\n--- Analise Tecnica ---")
print("Res A original: " + str(img_a_orig.shape[1]) + "x" + str(img_a_orig.shape[0]) + " = " + str(img_a_orig.shape[1]*img_a_orig.shape[0]) + " px")
print("Res A 50pct:     " + str(img_a_50.shape[1]) + "x" + str(img_a_50.shape[0]) + " = " + str(img_a_50.shape[1]*img_a_50.shape[0]) + " px")
print("Res A 20pct:     " + str(img_a_20.shape[1]) + "x" + str(img_a_20.shape[0]) + " = " + str(img_a_20.shape[1]*img_a_20.shape[0]) + " px")
print("Res B original: " + str(img_b_orig.shape[1]) + "x" + str(img_b_orig.shape[0]) + " = " + str(img_b_orig.shape[1]*img_b_orig.shape[0]) + " px")
print("Res B 50pct:     " + str(img_b_50.shape[1]) + "x" + str(img_b_50.shape[0]) + " = " + str(img_b_50.shape[1]*img_b_50.shape[0]) + " px")
print("Res B 20pct:     " + str(img_b_20.shape[1]) + "x" + str(img_b_20.shape[0]) + " = " + str(img_b_20.shape[1]*img_b_20.shape[0]) + " px")

## 4. B. Espaco de Cor

O espaco de cor determina como as informacoes de cor sao representadas numericamente. Os principais espacos sao:

- **RGB:** Combinacao aditiva de Vermelho, Verde e Azul — padrao para displays.
- **HSV:** Matiz (Hue), Saturacao e Valor — mais intuitivo para segmentacao baseada em cor.
- **Escala de Cinza:** Intensidade luminosa apenas — reduz dimensionalidade, util para deteccao de bordas e formas.

**Questoes para reflexao:**
- Quais diferencas visuais aparecem ao converter?
- Em que tipo de tarefa voce usaria cada uma?

**Respostas esperadas:**
- RGB e o padrao de entrada para redes neurais convolucionais.
- HSV facilita segmentacao por cor (ex: detectar objetos rosas independente da iluminacao).
- Escala de cinza reduz custo computacional e e suficiente para tarefas de forma/texture.

In [ ]:
# Espaco de Cor — Conversoes
img_a_bgr = imgs_a[0].copy()
img_b_bgr = imgs_b[0].copy()

# Conversoes
img_a_rgb = cv2.cvtColor(img_a_bgr, cv2.COLOR_BGR2RGB)
img_a_hsv = cv2.cvtColor(img_a_bgr, cv2.COLOR_BGR2HSV)
img_a_gray = cv2.cvtColor(img_a_bgr, cv2.COLOR_BGR2GRAY)

img_b_rgb = cv2.cvtColor(img_b_bgr, cv2.COLOR_BGR2RGB)
img_b_hsv = cv2.cvtColor(img_b_bgr, cv2.COLOR_BGR2HSV)
img_b_gray = cv2.cvtColor(img_b_bgr, cv2.COLOR_BGR2GRAY)

# Salvar
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img004_hsv.jpg"), cv2.cvtColor(img_a_hsv, cv2.COLOR_HSV2BGR))
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img005_gray.jpg"), img_a_gray)
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img004_hsv.jpg"), cv2.cvtColor(img_b_hsv, cv2.COLOR_HSV2BGR))
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img005_gray.jpg"), img_b_gray)

# Visualizar
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_a_rgb)
axes[0, 0].set_title("Classe A — RGB")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_a_hsv)
axes[0, 1].set_title("Classe A — HSV")
axes[0, 1].axis("off")

axes[0, 2].imshow(img_a_gray, cmap="gray")
axes[0, 2].set_title("Classe A — Escala de Cinza")
axes[0, 2].axis("off")

axes[1, 0].imshow(img_b_rgb)
axes[1, 0].set_title("Classe B — RGB")
axes[1, 0].axis("off")

axes[1, 1].imshow(img_b_hsv)
axes[1, 1].set_title("Classe B — HSV")
axes[1, 1].axis("off")

axes[1, 2].imshow(img_b_gray, cmap="gray")
axes[1, 2].set_title("Classe B — Escala de Cinza")
axes[1, 2].axis("off")

plt.suptitle("Analise de Espaco de Cor", fontsize=16)
plt.tight_layout()
plt.show()

# Analise
print("--- Analise das Conversoes de Cor ---")
print("RGB: Representacao direta das intensidades de cor (padrao para display e CNNs)")
print("HSV: Separa informacao de cor (Hue), intensidade (Saturation) e brilho (Value)")
print("     Util para segmentacao baseada em tonalidade")
print("Escala de Cinza: Reduz 3 canais para 1 — util quando cor nao e discriminante")
print("                Economia de memoria: reducao de 66pct nos dados")

## 4. C. Quantizacao

A quantizacao reduz o numero de niveis de intensidade (tons de cinza) de uma imagem. Isso simula cenarios de baixa profundidade de cor e permite avaliar a robustez de algoritmos.

**Niveis aplicados:**
- 256 niveis (original 8-bit)
- 64 niveis (6 bits)
- 32 niveis (5 bits)
- 2 niveis (1 bit — preto e branco puro)

**Questoes:**
- O que muda visualmente?
- Qual versao ainda seria util para o dataset e para qual aplicacao?

**Resposta esperada:**
- Com 256 > 64 niveis: degradacao quase imperceptivel; adequado para a maioria das tarefas.
- Com 32 niveis: leve posterizacao visivel; ainda util para classificacao geral.
- Com 2 niveis: perda severa; util apenas para tarefas de segmentacao binaria ou contornos.

In [ ]:
# Quantizacao — 1 imagem de cada classe
def quantize_image(img_gray, levels):
    """Reduz a imagem de escala de cinza para N niveis discretos."""
    factor = 256 // levels
    return (img_gray // factor) * factor

levels_list = [256, 64, 32, 2]
labels_list = ["256 (original)", "64 niveis", "32 niveis", "2 niveis (Binarizada)"]

# Quantizar ambas as imagens
quantized_a = [quantize_image(img_a_gray, lv) for lv in levels_list]
quantized_b = [quantize_image(img_b_gray, lv) for lv in levels_list]

# Salvar quantizacoes
for i in range(len(levels_list)):
    lv = levels_list[i]
    cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / ("img00" + str(6+i) + "_q" + str(lv) + ".jpg")), quantized_a[i])
    cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / ("img00" + str(6+i) + "_q" + str(lv) + ".jpg")), quantized_b[i])

# Visualizar
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx in range(4):
    axes[0, idx].imshow(quantized_a[idx], cmap="gray")
    axes[0, idx].set_title("Classe A — " + labels_list[idx])
    axes[0, idx].axis("off")

    axes[1, idx].imshow(quantized_b[idx], cmap="gray")
    axes[1, idx].set_title("Classe B — " + labels_list[idx])
    axes[1, idx].axis("off")

plt.suptitle("Analise de Quantizacao", fontsize=16)
plt.tight_layout()
plt.show()

print("--- Analise ---")
print("Quantizacao reduz a riqueza tonal da imagem.")
print("Niveis 256 e 64: praticamente identicos ao olho nu.")
print("Nivel 32: posterizacao sutil comeca a aparecer.")
print("Nivel 2 (binario): perda total de tonalidades — util apenas para segmentacao por limiarizacao.")
print("Para Visao Computacional, recomenda-se manter pelo menos 64 niveis.")

## 4. D. Formato de Arquivo

A escolha do formato de armazenamento impacta diretamente a qualidade e o tamanho dos dados.

| Formato | Caracteristica | Uso recomendado |
|---------|---------------|------------------|
| **JPEG** | Com perdas (lossy), compressao alta, artefatos visiveis em bordas | Datasets grandes, economia de espaco |
| **PNG** | Sem perdas (lossless), preserva todos os detalhes | Datasets de alta qualidade, mascaras, labels |

**Questoes para registrar:**
- Tamanho do arquivo (KB)
- Diferenca visual percebida
- Hipotese sobre impacto em tarefas de PDI

In [ ]:
# Formato de Arquivo — Comparativo JPEG vs PNG
import os

img_a_test = imgs_a[0].copy()
img_b_test = imgs_b[0].copy()

# Salvar JPEG com alta compressao (qualidade baixa para evidenciar artefatos)
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img010_jpeg_comprimido.jpg"), img_a_test, [cv2.IMWRITE_JPEG_QUALITY, 30])
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img010_jpeg_comprimido.jpg"), img_b_test, [cv2.IMWRITE_JPEG_QUALITY, 30])

# Salvar PNG sem perdas
cv2.imwrite(str(OUTPUT_DIR / "classe_A_rosa" / "img011_png_semperda.png"), img_a_test)
cv2.imwrite(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img011_png_semperda.png"), img_b_test)

# Carregar de volta e comparar
jpeg_a = cv2.imread(str(OUTPUT_DIR / "classe_A_rosa" / "img010_jpeg_comprimido.jpg"))
png_a = cv2.imread(str(OUTPUT_DIR / "classe_A_rosa" / "img011_png_semperda.png"))

jpeg_b = cv2.imread(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img010_jpeg_comprimido.jpg"))
png_b = cv2.imread(str(OUTPUT_DIR / "classe_B_nao_rosa" / "img011_png_semperda.png"))

# Tamanhos dos arquivos
size_jpeg_a = os.path.getsize(OUTPUT_DIR / "classe_A_rosa" / "img010_jpeg_comprimido.jpg") / 1024
size_png_a = os.path.getsize(OUTPUT_DIR / "classe_A_rosa" / "img011_png_semperda.png") / 1024
size_jpeg_b = os.path.getsize(OUTPUT_DIR / "classe_B_nao_rosa" / "img010_jpeg_comprimido.jpg") / 1024
size_png_b = os.path.getsize(OUTPUT_DIR / "classe_B_nao_rosa" / "img011_png_semperda.png") / 1024

# Diferenca (artefatos JPEG)
diff_a = cv2.absdiff(png_a, jpeg_a)
diff_b = cv2.absdiff(png_b, jpeg_b)

# Visualizar
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(cv2.cvtColor(png_a, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("PNG (sem perdas) — " + str(round(size_png_a, 1)) + " KB")
axes[0, 0].axis("off")

axes[0, 1].imshow(cv2.cvtColor(jpeg_a, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("JPEG (qual=30) — " + str(round(size_jpeg_a, 1)) + " KB")
axes[0, 1].axis("off")

axes[0, 2].imshow(cv2.cvtColor(diff_a, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title("Diferenca (PNG - JPEG)")
axes[0, 2].axis("off")

axes[1, 0].imshow(cv2.cvtColor(png_b, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("PNG (sem perdas) — " + str(round(size_png_b, 1)) + " KB")
axes[1, 0].axis("off")

axes[1, 1].imshow(cv2.cvtColor(jpeg_b, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("JPEG (qual=30) — " + str(round(size_jpeg_b, 1)) + " KB")
axes[1, 1].axis("off")

axes[1, 2].imshow(cv2.cvtColor(diff_b, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("Diferenca (PNG - JPEG)")
axes[1, 2].axis("off")

plt.suptitle("Comparativo: JPEG vs PNG — Artefatos de Compressao", fontsize=16)
plt.tight_layout()
plt.show()

print("--- Registro de Tamanhos ---")
print("Classe A — JPEG: " + str(round(size_jpeg_a, 1)) + " KB | PNG: " + str(round(size_png_a, 1)) + " KB")
print("Classe B — JPEG: " + str(round(size_jpeg_b, 1)) + " KB | PNG: " + str(round(size_png_b, 1)) + " KB")
print("")
print("--- Hipotese sobre impacto em PDI ---")
print("JPEG com alta compressao introduz artefatos de bloco (blocking artifacts)")
print("em bordas e regioes de alto contraste, o que pode degradar:")
print("  - Deteccao de bordas (Canny, Sobel)")
print("  - Segmentacao por cor")
print("  - Desempenho de modelos CNNs em classificacao fina")
print("PNG e preferivel para datasets de treino onde fidelidade e critica.")

## 5. Resumo do Mini Dataset Gerado

| Classe | Prefixo | Qtd. Imagens | Descricao das Transformacoes |
|--------|---------|--------------|-------------------------------|
| A (rosa) | img001–img011 | 11 | Original + 50pct, 20pct resolucao + HSV + Cinza + Quantizacoes (256,64,32,2) + JPEG + PNG |
| B (nao-rosa) | img001–img011 | 11 | Identico a classe A |

**Total de imagens:** 22 arquivos (11 por classe)  
**Estrutura:** `mini_dataset/classe_A_rosa/` e `mini_dataset/classe_B_nao_rosa/`

---

## 6. Analise Reflexiva

### 6.1 Resolucao
- A reducao de resolucao impacta diretamente a quantidade de informacao espacial. A 20pct, detalhes finos sao perdidos.
- Para classificacao de cenas (cenas amplas), a perda pode ser toleravel; para deteccao de objetos pequenos, e critica.

### 6.2 Espaco de Cor
- RGB e indispensavel para treino de CNNs padrao.
- HSV e superior para tarefas de segmentacao baseada em cor.
- Cinza: reduz dimensionalidade e pode ser usada para deteccao de formas/edges.

### 6.3 Quantizacao
- Niveis abaixo de 32 introduzem posterizacao visivel. Recomenda-se >=64 niveis para preservar qualidade.

### 6.4 Formato de Arquivo
- JPEG economiza espaco (~80-90pct menor que PNG) mas introduz artefatos.
- PNG e ideal para datasets de treino e anotacoes (mascaras, labels).

In [ ]:
# Gerar relatorio final em Markdown
relatorio = """
# Relatorio Final — Mini Dataset de Processamento de Imagens Digitais

## 1. Aquisicao de Imagens
- **Dispositivo:** Samsung A12 (camera traseira, modo automatico)
- **Iluminacao:** Noturna (luz artificial)
- **Distancia:** 10–25 cm
- **Angulos:** Frontal, ~30 graus, ~45 graus zenital
- **Observacao:** Ruido visivel em ISO alto; leve desfoque periferico

## 2. Classes
| Classe | Descricao | Imagens Originais |
|--------|-----------|-------------------|
| A | Objetos com cor dominante rosa | 5 |
| B | Objetos com cor dominante nao rosa | 5 |

## 3. Transformacoes Aplicadas
| Transformacao | Classe A | Classe B |
|---------------|----------|----------|
| Original | sim | sim |
| 50pct Resolucao | sim | sim |
| 20pct Resolucao | sim | sim |
| HSV | sim | sim |
| Escala de Cinza | sim | sim |
| Quantizacao 256 | sim | sim |
| Quantizacao 64 | sim | sim |
| Quantizacao 32 | sim | sim |
| Quantizacao 2 (binario) | sim | sim |
| JPEG comprimido | sim | sim |
| PNG sem perdas | sim | sim |

## 4. Analise Tecnica

### 4.1 Resolucao
A reducao de resolucao impacta diretamente a quantidade de informacao espacial. A 20pct, detalhes finos sao perdidos.

### 4.2 Espaco de Cor
- RGB: padrao para redes neurais
- HSV: melhor para segmentacao por cor
- Cinza: economico e eficaz para tarefas baseadas em forma

### 4.3 Quantizacao
Niveis abaixo de 32 introduzem posterizacao visivel. Recomenda-se >=64 niveis.

### 4.4 Formato
JPEG 30pct qualidade gera artefatos perceptiveis. PNG e preferivel quando fidelidade e prioridade.

## 5. Conclusao
O dataset gerado cobre os fundamentos de aquisicao, resolucao, cor, quantizacao e formatos de arquivo. A estrutura segue o padrao de diretorios para ML, pronto para uso com ferramentas como ImageDataGenerator, torchvision.datasets.ImageFolder ou tf.keras.utils.image_dataset_from_directory.
"""

relatorio_path = OUTPUT_DIR / "RELATORIO.md"
with open(relatorio_path, "w", encoding="utf-8") as f:
    f.write(relatorio.strip())

print("Relatorio salvo em:", relatorio_path)
print("\n--- Conteudo do Relatorio ---")
print(relatorio)

In [ ]:
# Listagem final do dataset gerado
import os

print("=" * 60)
print("ESTRUTURA FINAL DO MINI DATASET")
print("=" * 60)

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), "").count(os.sep)
    indent = " " * 2 * level
    print(indent + os.path.basename(root) + "/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(subindent + file + " (" + str(round(size_kb, 1)) + " KB)")

print("")
print("=" * 60)
print("Dataset pronto para uso em Visao Computacional!")
print("=" * 60)